# Clinical Efficacy of Cemiplimab (Libtayo®) in Advanced & High-Risk Cutaneous Squamous Cell Carcinoma (cSCC)

This notebook performs reproducible clinical data analysis and visualization for the pivotal clinical trials of cemiplimab in cutaneous squamous cell carcinoma (cSCC):
1. **EMPOWER-CSCC-1 (NCT02760498)**: A Phase 2 open-label study evaluating cemiplimab monotherapy in advanced cSCC (metastatic and locally advanced).
2. **C-POST (NCT03969004)**: A Phase 3 randomized, double-blind study evaluating adjuvant cemiplimab vs placebo in high-risk cSCC patients after surgery and radiation.

## Setup and Imports

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set style for publication-quality figures
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.size': 11,
    'axes.labelsize': 12,
    'axes.titlesize': 14,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'figure.titlesize': 16,
    'savefig.bbox': 'tight',
    'savefig.dpi': 300
})

# Define consistent color palette
PALETTE = {
    'cemiplimab': '#1f77b4',  # Sleek blue
    'placebo': '#7f7f7f',     # Neutral gray
    'cr': '#2ca02c',          # Green for Complete Response
    'pr': '#bcbd22',          # Olive for Partial Response
    'non_resp': '#ff7f0e'     # Orange for Non-responders
}

# Create figures directory if it doesn't exist
os.makedirs('../figures', exist_ok=True)
print("Setup complete. Package versions:")
print(f"Pandas: {pd.__version__}")
print(f"Seaborn: {sns.__version__}")

## Part 1: EMPOWER-CSCC-1 Efficacy Analysis
We load the efficacy dataset containing the Objective Response Rate (ORR), Complete Response (CR) Rate, and Partial Response (PR) Rate across the trial cohorts.

In [ ]:
# Load EMPOWER-CSCC-1 dataset
empower_df = pd.read_csv('../data/empower_cscc_1_efficacy.csv')
empower_df

### Plot 1: Objective Response Rates (ORR) with CR and PR Breakdown
We construct a stacked bar chart to show the proportions of patients achieving CR and PR, summing up to the total ORR, for each cohort.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

cohorts = empower_df['cohort_id']
cr_rates = empower_df['cr_pct']
pr_rates = empower_df['pr_pct']
non_resp_rates = 100 - empower_df['orr_pct']

# Stacked bar chart
bar_width = 0.5
r = np.arange(len(cohorts))

# Draw bars
bars_cr = ax.bar(r, cr_rates, color=PALETTE['cr'], edgecolor='black', width=bar_width, label='Complete Response (CR)')
bars_pr = ax.bar(r, pr_rates, bottom=cr_rates, color=PALETTE['pr'], edgecolor='black', width=bar_width, label='Partial Response (PR)')

# Add labels and text
ax.set_ylabel('Percentage of Patients (%)', fontweight='bold')
ax.set_title('EMPOWER-CSCC-1: Objective Response Rates (ORR) by Cohort\n(Independent Central Review)', fontweight='bold', pad=15)
ax.set_xticks(r)
ax.set_xticklabels(cohorts, rotation=15, ha='right')
ax.set_ylim(0, 70)

# Add data annotations
for i in range(len(cohorts)):
    orr = empower_df.loc[i, 'orr_pct']
    n = empower_df.loc[i, 'n_evaluable']
    cr = empower_df.loc[i, 'cr_pct']
    pr = empower_df.loc[i, 'pr_pct']
    
    # Label for ORR on top of the stacked bar
    ax.text(i, orr + 1.5, f"ORR: {orr}%\n(n={n})", ha='center', va='bottom', fontweight='bold', fontsize=10)
    # Labels inside the segments
    ax.text(i, cr/2, f"{cr}%", ha='center', va='center', color='white', fontweight='bold', fontsize=9)
    ax.text(i, cr + pr/2, f"{pr}%", ha='center', va='center', color='black', fontweight='bold', fontsize=9)

ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig('../figures/empower_cscc_1_efficacy.png', dpi=300)
plt.show()

## Part 2: C-POST Adjuvant Efficacy Analysis
We load the disease-free survival (DFS) dataset from the C-POST trial, which compares adjuvant cemiplimab vs placebo in high-risk patients.

In [ ]:
# Load C-POST dataset
cpost_df = pd.read_csv('../data/c_post_efficacy.csv')
cpost_df

### Plot 2: Disease-Free Survival (DFS) Rates at 12 & 24 Months
We create a grouped bar chart to visualize DFS rates at 12 and 24 months for adjuvant cemiplimab vs placebo.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

x = np.arange(2)  # 12-month and 24-month time points
width = 0.35

cemi_dfs = [cpost_df.loc[cpost_df['trial_arm']=='Cemiplimab', 'dfs_rate_12m'].values[0],
            cpost_df.loc[cpost_df['trial_arm']=='Cemiplimab', 'dfs_rate_24m'].values[0]]
placebo_dfs = [cpost_df.loc[cpost_df['trial_arm']=='Placebo', 'dfs_rate_12m'].values[0],
               cpost_df.loc[cpost_df['trial_arm']=='Placebo', 'dfs_rate_24m'].values[0]]

rects1 = ax.bar(x - width/2, cemi_dfs, width, label='Adjuvant Cemiplimab (n=209)', color=PALETTE['cemiplimab'], edgecolor='black')
rects2 = ax.bar(x + width/2, placebo_dfs, width, label='Placebo (n=206)', color=PALETTE['placebo'], edgecolor='black')

ax.set_ylabel('Disease-Free Survival Rate (%)', fontweight='bold')
ax.set_title('C-POST Trial: Disease-Free Survival (DFS) Rates at 12 & 24 Months\n(Adjuvant cSCC Setting)', fontweight='bold', pad=15)
ax.set_xticks(x)
ax.set_xticklabels(['12-Month DFS Rate', '24-Month DFS Rate'], fontweight='bold')
ax.set_ylim(0, 110)
ax.legend(loc='lower left')

# Annotate bars
def autolabel(rects):
    for rect in rects:
        height = rect.get_height()
        ax.annotate(f'{height}%',
                    xy=(rect.get_x() + rect.get_width() / 2, height),
                    xytext=(0, 3),  # 3 points vertical offset
                    textcoords="offset points",
                    ha='center', va='bottom', fontweight='bold')

autolabel(rects1)
autolabel(rects2)

plt.tight_layout()
plt.savefig('../figures/c_post_dfs_efficacy.png', dpi=300)
plt.show()

### Plot 3: C-POST Disease-Free Survival Hazard Ratio (Forest Plot)
We visualize the treatment effect size (Hazard Ratio) and its precision (95% confidence interval) indicating the risk reduction of recurrence or death.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))

hr = cpost_df.loc[cpost_df['trial_arm']=='Cemiplimab', 'hazard_ratio'].values[0]
lower = cpost_df.loc[cpost_df['trial_arm']=='Cemiplimab', 'hr_ci_lower'].values[0]
upper = cpost_df.loc[cpost_df['trial_arm']=='Cemiplimab', 'hr_ci_upper'].values[0]

# Plot hazard ratio and CI
ax.errorbar(hr, 1, xerr=[[hr - lower], [upper - hr]], fmt='o', color=PALETTE['cemiplimab'], 
            markersize=10, elinewidth=3, capsize=8, label=f'Cemiplimab vs Placebo (HR: {hr})')

# Vertical line at HR = 1.0 (no treatment effect)
ax.axvline(1.0, color='red', linestyle='--', linewidth=1.5, label='No Effect (HR = 1.0)')

ax.set_yticks([])
ax.set_ylim(0.5, 1.5)
ax.set_xlim(0.0, 1.2)
ax.set_xlabel('Hazard Ratio (HR) for Recurrence or Death (95% CI)', fontweight='bold')
ax.set_title('C-POST Trial: Disease-Free Survival Hazard Ratio', fontweight='bold', pad=15)

# Add text annotation
ax.text(hr, 1.15, f"HR: {hr} (95% CI: {lower} - {upper})\nP < 0.0001\n68% Risk Reduction", 
        ha='center', va='bottom', fontweight='bold', color='#1f77b4', bbox=dict(facecolor='white', alpha=0.8, edgecolor='none'))

ax.text(0.1, 0.7, 'Favors Adjuvant Cemiplimab', ha='left', va='center', fontweight='bold', color='green')
ax.text(1.1, 0.7, 'Favors Placebo', ha='right', va='center', fontweight='bold', color='red')

plt.tight_layout()
plt.savefig('../figures/c_post_hazard_ratio.png', dpi=300)
plt.show()